# Regularizacion: Ridge y Lasso

La **regularizacion** agrega una penalidad a la funcion de costo (RSS) para achicar los coeficientes hacia cero y controlar el sobreajuste:

$$CF = \sum_{i=1}^{N}(\hat{y}_i - y_i)^2 + \alpha \cdot \text{(penalidad sobre } \beta)$$

- **Ridge** usa la norma **L2** (`Σ β²`): achica todos los coeficientes pero ninguno llega exactamente a 0.
- **Lasso** usa la norma **L1** (`Σ |β|`): puede llevar coeficientes **exactamente a 0**, haciendo seleccion de variables (modelos dispersos).

El hiperparametro `alpha` (la `λ` de la teoria) regula la fuerza de la penalidad y se elige por **validacion cruzada**.

Usamos el dataset `Credit` para predecir `Balance`. A proposito agregamos una variable **redundante** (`Rating_2 = 2·Rating`, perfectamente colineal con `Rating`) para ver como cada metodo maneja la redundancia.

In [ ]:
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Datos

`Credit`: predecimos `Balance` (deuda en la tarjeta) a partir de variables del cliente.

In [ ]:
import pandas as pd

data = pd.read_csv('../datasets/Credit.csv')
data.head()

## Preparacion: variable redundante + estandarizacion

Creamos `Rating_2 = 2·Rating` (colineal con `Rating`) para el experimento.

**Estandarizamos** con `StandardScaler` porque Ridge y Lasso son **sensibles a la escala**: la penalidad se calcula sobre los coeficientes, asi que una variable con valores mas grandes pesaria distinto en la penalizacion solo por su unidad. Tras estandarizar, cada variable queda en unidades de su propio desvio estandar.

In [ ]:
data['Rating_2']=2*data.Rating
X=data[['Income','Limit','Rating','Age','Rating_2']]
scaler=StandardScaler()
X_std=scaler.fit_transform(X)
y=data['Balance']

## Ridge (L2): eleccion de alpha por CV

`RidgeCV` prueba una grilla de valores de `alpha` y elige el mejor por validacion cruzada (5 folds). `alpha_` es el ganador y `best_score_` su R2 promedio en CV.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.3, random_state=127)

model_ridge_cv = linear_model.RidgeCV(
    alphas=[0.3, 0.5, 1.0, 1.1, 1.15, 1.17, 1.18, 1.19, 1.2, 1.21, 1.22, 1.3, 1.4, 1.5, 10.0],
    fit_intercept=True,
    cv=5
)

model_fit_ridge_cv = model_ridge_cv.fit(X_train, y_train)

print(model_fit_ridge_cv.alpha_)
print(getattr(model_fit_ridge_cv, "best_score_", None))

In [ ]:
best_alpha=model_fit_ridge_cv.alpha_
model_ridge=linear_model.Ridge(alpha=best_alpha,fit_intercept=True)
model_fit_ridge=model_ridge.fit(X_train,y_train)
print(model_fit_ridge.coef_)
print(model_fit_ridge.intercept_)
print(model_fit_ridge.score(X_train,y_train))

Notar que `Rating` y `Rating_2` reciben **el mismo coeficiente**: Ridge reparte el peso por igual entre las dos variables colineales (la penalidad L2 es minima cuando el peso se divide en partes iguales).

In [ ]:
model_fit_ridge.score(X_test,y_test)

## Lasso (L1): seleccion de variables

`LassoCV` elige `alpha` por CV (10 folds). A diferencia de Ridge, Lasso puede anular coeficientes.

In [ ]:
model_lasso_cv=linear_model.LassoCV(alphas=[0.05,0.06,0.07,0.08,0.09,0.95,0.97,0.1,0.12,0.15,0.2,0.3,1.0,10.0],fit_intercept=True,cv=10)

model_fit_lasso_cv=model_lasso_cv.fit(X_train,y_train)

print(model_fit_lasso_cv.alpha_)
print(model_fit_lasso_cv.score(X_train,y_train))

In [ ]:
best_alpha=model_fit_lasso_cv.alpha_
model_lasso=linear_model.Lasso(alpha=best_alpha,fit_intercept=True)
model_fit_lasso=model_lasso.fit(X_train,y_train)
print(model_fit_lasso.coef_)
print(model_fit_lasso.intercept_)
print(model_fit_lasso.score(X_test,y_test))

Aca se ve la diferencia clave: en vez de repartir, Lasso **concentra el peso en `Rating` y empuja `Rating_2` casi a 0**. La penalidad L1 favorece soluciones dispersas: ante variables redundantes, se queda con una y descarta la otra (seleccion de variables).

## Comparacion con OLS sin regularizar

La regresion lineal comun, como referencia. Al igual que Ridge, reparte el peso por igual entre las dos colineales; el R2 de test queda muy parecido al de los modelos regularizados porque el dataset tiene poco ruido y muchas observaciones.

In [ ]:
model=linear_model.LinearRegression()
model_fit=model.fit(X_train,y_train)
print(model_fit.coef_)
print(model_fit.intercept_)
print(model_fit.score(X_test,y_test))